In [ ]:
from xsimlab.model import _ModelBuilder, filter_variables
from xsimlab.variable import VarType
import numpy as np


def fill_value_from_dtype(dtype=None):
    """ Try to keep NaN for not yet appeard or pruned GUs """
    if dtype is None:
        return 0.
    if dtype.kind in 'f':
        return 0.
    elif dtype.kind == 'M':
        return np.datetime64('NaT')
    elif dtype.kind == 'O':
        return None
    elif dtype.kind == 'U':
        return ''
    else:
        return 0.


class State(dict):

    indices = {}
    variables = {}

    def __init__(self, variables, indices):
        self.variables = variables
        self.indices = indices
        dict.__init__(self)

    def resize(self, index_name, new_shape):
        '''Resize a variable value if a 1D index  in the variables dimensions increased in length.
        '''
        
        for var_name, var in self.variables.items():
            #print(var_name, var)
            if var_name in self and type(self[var_name]) is np.ndarray:
                var_value = self[var_name]
                var_shape = var_value.shape
                var_dims = np.array(var.metadata.get('dims')).flatten()
                #print(index_name, var_name, var_dims, len(var_dims), index_name in var_dims)
                if len(var_dims) and index_name in var_dims:
                    fill_value = None
                    var_enc = var.metadata.get('encoding')
                    if 'fill_value' in var_enc:
                        fill_value = var_enc['fill_value']
                    else:
                        fill_value = fill_value_from_dtype(var_value.dtype)
                    var_shape = (var_dims == index_name) * new_shape + (var_dims != index_name) * var_shape
                    if (tuple(var_shape) != var_value.shape):
                        if len(var_shape) == 1:
                            data = np.empty(var_shape, dtype=var_value.dtype)
                            data[0:var_value.shape[0]] = var_value
                            data[var_value.shape[0]:] = fill_value
                        else:
                            data = np.full(var_shape, fill_value, dtype=var_value.dtype)
                            data[tuple([slice(0, i) for i in var_value.shape])] = var_value
                        #print(var_name, data)
                        super(State, self).__setitem__(var_name, data)

    def __setitem__(self, item, new):
        if item in self and item in self.indices and self[item].shape != new.shape:
            index = self.indices[item]
            var_dims = np.array(index.metadata.get('dims')).flatten()
            if len(var_dims) == 1:
                self.resize(var_dims[0], new.shape)
            else:
                print('Multi dim indices not supported in resizing')
        super(State, self).__setitem__(item, new)


def set_state(self):
    variables = {}
    indices = {}
    non_indices = {}
    for p_name, p_obj in self._processes_obj.items():
        for v_name, variable in filter_variables(p_obj).items():
            if variable.metadata['var_type'] == VarType.INDEX:
                indices[(p_name, v_name)] = variable
            else:
                non_indices[(p_name, v_name)] = variable
            variables[(p_name, v_name)] = variable

    state = State(non_indices, indices)

    # bind state to each process in the model
    for p_obj in self._processes_obj.values():
        p_obj.__xsimlab_state__ = state

    return state


# 'patch' xsimlab to use a custom state class instead of a dict
_ModelBuilder.set_state = set_state

In [ ]:
import xsimlab as xs
import openalea.lpy as lpy
import numpy as np
import pandas as pd
import attr
import pathlib

import vmapplet

In [ ]:
_Model = xs.Model
class Model(_Model):
    
    __lpyfied_processes__ = tuple()
    
    def __init__(self, processes):
        self.__lpyfied_processes__ = tuple([process for process in processes.values() if hasattr(process, '__is__lpyfied_process__')])
        super().__init__(processes)
        
        
    def _execute_process(
        self, p_obj, stage, runtime_context, hooks, validate, state=None
    ):
        """Internal process execution method, which calls the process object's
        executor.
        A state may be passed to the executor instead of using the executor's
        state (this is to avoid stateful objects when calling the executor
        during execution of a Dask graph).
        The process executor returns a partial state (only the variables that
        have been updated by the executor, which will be needed for executing
        further tasks in the Dask graph).
        This method returns this updated state as well as any runtime signal returned
        by the hook functions and/or the executor (the one with highest priority).
        """
        
        if stage != xs.model.SimulationStage.INITIALIZE:
            # ignore read/write access during initalization - for now
            # if process.__context_process__ is None then stage is initalization
            for process in self.__lpyfied_processes__:
                process.__context_process__ = type(p_obj)
        
        executor = p_obj.__xsimlab_executor__
        p_name = p_obj.__xsimlab_name__

        signal_pre = self._call_hooks(hooks, runtime_context, stage, "process", "pre")

        if signal_pre.value > 0:
            return p_name, ({}, signal_pre)

        state_out, signal_out = executor.execute(
            p_obj, stage, runtime_context, state=state
        )

        signal_post = self._call_hooks(hooks, runtime_context, stage, "process", "post")
        if signal_post.value > signal_out.value:
            signal_out = signal_post

        if validate:
            self.validate(self._processes_to_validate[p_name])
            
        if stage == xs.model.SimulationStage.INITIALIZE:
            # if the process has an Lsystem variable (any_object) register the read and read-write variables in the lpyfied process
            if len([name for name in xs.filter_variables(p_obj, var_type=xs.model.VarType.OBJECT).keys() if type(getattr(p_obj, name)) == lpy.Lsystem]):
                variables = {}
                for var_name, var in xs.filter_variables(p_obj).items():
                    var_meta = var.metadata
                    for lpyfied_process in self.__lpyfied_processes__:
                        dim  = lpyfied_process.__lpyfied_dim__
                        if 'dims' in var_meta and var_meta['dims'][0] == (dim,):
                            print(type(p_obj), var_name)
                            var_type = var_meta['var_type']
                            if var_type == xs.model.VarType.VARIABLE:
                                variables[var_name] = 1  # read-write
                            elif var_type == xs.model.VarType.FOREIGN:
                                variables[var_meta['var_name']] = 0  # read-only, var_name from foreign process
                        if type(p_obj) in lpyfied_process.__lsystem_processes__:
                            lpyfied_process.__lsystem_processes__[type(p_obj)].update(variables)
                        else:
                            lpyfied_process.__lsystem_processes__[type(p_obj)] = {**variables}
                    

        return p_name, (state_out, signal_out)        
        
xs.Model = Model

In [ ]:
class DotDict:
    
    __process = None
    __data = None
    __index_name = ''
    
    def __init__(self, process, data, index_name):
        self.__process = process
        self.__data = data
        self.__index_name = index_name
        
    def __getattr__(self, arg):
        
        # index is always available but should be read-only
        if arg == self.__index_name or self.__process.__context_process__ is None:  # initalization
            return self.__data[arg]
        
        if self.__process.__context_process__ in self.__process.__lsystem_processes__:
            item = self.__data[arg]
            # is not the process itself
            if self.__process.__context_process__ != type(self.__process):
                variables = self.__process.__lsystem_processes__[self.__process.__context_process__]
                if arg in variables:
                    if type(item) == np.array:
                        item.flags.writeable = bool(variables[arg])
                else:
                    item = None
        return item
        
def lpyfy(dim):
    #  use attr name? what to do if index in foreign? global?
    def wrap(cls):
        import xsimlab as xs
        
        VARIABLE = xs.model.VarType.VARIABLE
        INDEX = xs.model.VarType.INDEX
        # raise error if dim/coord is not in cls or dim is None and there are multiple coords
        
        # cache index and relevant attribute names and var_type
        attributes = {}
        index_name = None
        for name, var in xs.filter_variables(cls).items():
            meta = var.metadata
            if 'dims' in meta and meta['dims'][0] == (dim,) and meta['var_type'] in (VARIABLE, INDEX):
                attributes[name] = meta['var_type']
                if meta['var_type'] == INDEX:
                    index_name = name
        
        def __iter__(self):
            return self
        
        def __next__(self):
            try:
                item = self[self.__current_index___]
            except IndexError:
                self.__current_index___ = 0
                raise StopIteration()
            self.__current_index___ += 1
            return item

        def __getitem__(self, arg):
            
            #print(inspect.stack()[-1])
            #print(globals())
            arg_type = type(arg)
            # test if index, slice or mask
            if np.issubdtype(arg_type, int):  # or arg_type == slice or (arg_type == np.ndarray and arg.dtype == bool)
                data = {}
                for name, var_type in attributes.items():
                    value = self[name]
                    if var_type == INDEX:
                        data[name] = value[arg]
                    else:
                        if type(value) == np.ndarray:
                            # return view
                            data[name] = value[arg:arg+1]
                        else:
                            data[name] = value
                return DotDict(self, data, index_name)
            elif arg_type == str:
                return getattr(self, arg)
            
        def new(self):
            if index_name:
                index = self[index_name]
                new_index = index.max() + 1
                setattr(self, index_name, np.concatenate((index, [new_index])))
                return self[new_index]
            
            
        
        cls.__iter__ = __iter__
        cls.__next__ = __next__
        cls.__getitem__ = __getitem__
        cls.__current_index___ = 0
        cls.__lsystem_processes__ = {}
        cls.__context_process__ = None
        cls.__is__lpyfied_process__ = True
        cls.__lpyfied_dim__ = dim
        
        setattr(cls, f'new_{dim}', new)
        
        return cls
        
    return wrap
        
    

In [ ]:
@lpyfy('metamer')
@xs.process
class Topology:
    
    lsystem = xs.any_object()
    
    index = xs.index('metamer', global_name='metamer')
    
    # in
    inital_sequence = xs.variable(dims='initial_metamer', static=True)
    
    # out
    apex_sequence_len = xs.variable(dims='metamer', intent='out')
    lstring = xs.any_object('lstring')
    radius = xs.variable(dims='metamer', intent='out')
    
    @xs.runtime(args=('nsteps'))
    def initialize(self, nsteps):
        self.index = np.arange(self.inital_sequence.size)
        self.apex_sequence_len = np.full(self.inital_sequence.shape, 0.)
        self.radius = np.full(self.inital_sequence.shape, 0.01)
        
        self.lsystem = lpy.Lsystem('topology.lpy', {
            'process': self,
            'derivation_length': int(nsteps)
        })
        
        self.lstring = self.lsystem.derive(self.lsystem.axiom, 0, 1)
        
    @xs.runtime(args=('step'))
    def run_step(self, step):
        
        self.lstring = self.lsystem.derive(self.lstring, step, 1)
        #print(self.lstring)
        

In [ ]:
@xs.process
class Phenology:
    
    # foreign
    index = xs.global_ref('metamer')
    
    # out
    apex_observation = xs.variable(dims='metamer', intent='out')
    
    def initialize(self):

        self.apex_observation = np.full(self.index.shape, 'dormant', dtype='<U20')
    

In [ ]:
from vmapplet.sequences import generate_sequence

@xs.process
class TopologicalDevelopment:
    
    # in
    bud_break_doy = xs.variable(static=True)
    lstring = xs.global_ref('lstring')
    
    # foreign
    apex_observation = xs.foreign(Phenology, 'apex_observation')
    apex_sequence_len = xs.foreign(Topology, 'apex_sequence_len')
    
    # out
    #apex_new_sequence_len = xs.variable(dims='metamer', intent='inout')
    
    
    @xs.runtime(args=['step_start'])
    def run_step(self, step_start):
        pass
        #print(step_start, self.apex_observation)


In [ ]:
@xs.process
class Geometry:
    
    lsystem = xs.any_object()
    geom_lstring = xs.any_object()
    scene = xs.any_object()
    
    lstring = xs.global_ref('lstring')
    
    apex_sequence_len = xs.foreign(Topology, 'apex_sequence_len')
    radius = xs.foreign(Topology, 'radius')
    
    @xs.runtime(args=('nsteps'))
    def initialize(self, nsteps):

        self.lsystem = lpy.Lsystem('geometry.lpy', {
            'process': self,
            'derivation_length': int(nsteps)
        })
        
        self.geom_lstring = self.lsystem.derive(self.lstring, 0, 1)
        self.scene = self.lsystem.sceneInterpretation(self.geom_lstring)
        
    @xs.runtime(args=('step'))
    def run_step(self, step):
        
        self.geom_lstring = self.lsystem.derive(self.lstring, step, 1)
        self.scene = self.lsystem.sceneInterpretation(self.geom_lstring)
        

In [ ]:
model = xs.Model({
    'topo': Topology,
    'topo_dev': TopologicalDevelopment,
    'pheno': Phenology,
    'geo': Geometry
})

In [ ]:
model.visualize()

In [ ]:
setup = xs.create_setup(
    model,
    clocks={
        'day': pd.date_range('2012-04-01', periods=10, freq='D')
    },
    master_clock='day',
    input_vars={
        'topo__inital_sequence': [0],
        'topo_dev__bud_break_doy': 135
    },
    output_vars={
        'pheno__apex_observation': 'day',
        'topo__radius': 'day'
    }
)

In [ ]:
from pgljupyter import SceneWidget

sw = SceneWidget()

@xs.runtime_hook(stage='run_step')
def hook(model, context, state):
    scene = state[('geo', 'scene')]
    if scene is not None:
        sw.set_scenes(scene, scales=1/10)

In [ ]:
sw

In [ ]:
ds = setup.xsimlab.run(model, decoding=dict(mask_and_scale=False), hooks=[hook])

In [ ]:
ds

In [ ]:
ds.topo__radius.plot(col='metamer', col_wrap=6)